# Parking V1 — prepare training inputs on CPU
**PARKING-V1-TRAINING-INPUTS-CPU-2026-10-29-r1**

Choose **Runtime → Change runtime type → Hardware accelerator: None**. This notebook prepares data only. It does not train, import a model, change V0, open held-out sensor samples, or update the iPhone.

V0's earlier training labels started at 2 m. This new package includes measured object-surface depth below 2 m, while keeping the existing frozen training/dev/held-out recording logs. It selects at most 4,096 training views and reports actual coverage for Vehicle/Pedestrian at 0–2, 2–4 and 4–6 m. Sparse holes remain unlabelled.

## Before starting — avoid repeated small Drive reads
The completed coverage **folder** is `MyDrive/mobile_adas3d_outputs/parking_near_field_coverage/nuscenes_full_0_6m_r1`. On drive.google.com, download that folder as a ZIP, then upload the ZIP to Colab's `/content` Files panel. It contains coverage metadata/chunks, **not the 45 GB sensor dataset**. Upload every ZIP volume if Drive splits it; retain filenames beginning `nuscenes_full_0_6m_r1`. The small `nuscenes_0_6m_coverage_results.zip` alone is not sufficient.

Also upload the unchanged reviewed `download_manifest_keyframes_all.json` if mounted Drive cannot read it. A previously completed `source_state.zip` in this run is automatically reusable and needs no folder upload.

If Drive is healthy, you may explicitly set `ALLOW_ONE_TIME_SMALL_DRIVE_READS=True` instead of uploading the folder. This permits one initial scan of 537 coverage chunks, then caches them locally and in one archive. It does **not** bypass quotas. I/O error 5 stops without automatic retries or remounts.

The sensor-preparation stage may sequentially read all ten source keyframe archives (about 45 GB). It keeps one source part at a time, and saves one completed training archive per part. Leave the accelerator off throughout.


## 1. CPU settings and main checkout
Keep the same `RUN_ID` to resume. Do not delete original datasets, completed runs or the frozen benchmark. Logs stay on the Colab VM; finished training-part archives are saved to Drive.

In [ ]:
from pathlib import Path
from collections import deque
import datetime, json, os, shlex, subprocess, sys
REVISION = 'PARKING-V1-TRAINING-INPUTS-CPU-2026-10-29-r1'
print(REVISION, flush=True)
if os.environ.get('COLAB_GPU', '').lower() not in ('', '0', 'none'):
    raise RuntimeError('Select Hardware accelerator: None. Do not pay for an idle GPU during preparation.')
from google.colab import drive
drive.mount('/content/drive')
RUN_ID = 'parking_v1_six_cell_training_inputs_r1'
if not RUN_ID or any(ch not in 'abcdefghijklmnopqrstuvwxyzABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789_-' for ch in RUN_ID):
    raise ValueError('Invalid RUN_ID')
MOBILE = Path('/content/mobile_adas3d')
DATASET = Path('/content/drive/MyDrive/mobile_adas3d_datasets/nuscenes_v1_0')
COVERAGE = Path('/content/drive/MyDrive/mobile_adas3d_outputs/parking_near_field_coverage/nuscenes_full_0_6m_r1')
OUTPUT = Path('/content/drive/MyDrive/mobile_adas3d_outputs/parking_v1_training_inputs') / RUN_ID
LOCAL = Path('/content') / ('parking_v1_inputs_' + RUN_ID)
LOGS = Path('/content') / ('parking_v1_logs_' + RUN_ID)
DEPS = Path('/content/parking_v1_preparation_cpu_deps')
PYTHON = Path(sys.executable)
COVERAGE_UPLOADS = sorted(Path('/content').glob('nuscenes_full_0_6m_r1*.zip'))
DOWNLOAD_MANIFEST_UPLOAD = Path('/content/download_manifest_keyframes_all.json')
ALLOW_ONE_TIME_SMALL_DRIVE_READS = False
WORKERS = 4
READY = False

def run(command, name, cwd=None, task_env=True):
    command = [str(x) for x in command]
    stamp = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
    path = LOGS / f'{name}_{stamp}.log'; path.parent.mkdir(parents=True, exist_ok=True)
    env = os.environ.copy(); env['CUDA_VISIBLE_DEVICES']=''
    if task_env:
        env.update(PYTHONPATH=os.pathsep.join([str(DEPS), str(MOBILE)]), OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1', MKL_NUM_THREADS='1')
    print('+', shlex.join(command), '\nLOCAL log:', path, flush=True)
    tail = deque(maxlen=25)
    with path.open('w') as stream:
        process = subprocess.Popen(command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True); stream.write(line); stream.flush(); tail.append(line.rstrip())
            code = process.wait()
        except BaseException:
            process.terminate(); process.wait(); raise
        finally:
            process.stdout.close()
        stream.write(f'\nEXIT CODE: {code}\n')
    if code: raise RuntimeError(f'Exit {code}; preserve files and inspect {path}\n'+'\n'.join(tail))
    return path

url = '/'.join(['https:', '', 'github.com', 'Ali-RT', 'mobile_adas3d.git'])
if not MOBILE.exists(): run(['git','clone','--branch','main',url,MOBILE], 'clone', task_env=False)
branch = subprocess.check_output(['git','branch','--show-current'], cwd=MOBILE, text=True).strip()
if branch != 'main': raise RuntimeError(f'Use a clean main checkout; preserve branch {branch!r}')
dirty = subprocess.check_output(['git','status','--porcelain','--untracked-files=no'], cwd=MOBILE, text=True)
if dirty.strip(): raise RuntimeError('Tracked local changes exist. Preserve them; no reset or overwrite is performed.')
run(['git','pull','--ff-only'], 'update_main', MOBILE, task_env=False)
print('Local coverage uploads:', COVERAGE_UPLOADS)
print('Read-only dataset:', DATASET, '\nNew output:', OUTPUT, '\nLocal scratch:', LOCAL)


## 2. Small CPU dependencies and tests
No Torch, CUDA, teacher model or custom extension is installed. This step checks the preparation contract before accessing sensor archives.

In [ ]:
SCRIPT = MOBILE/'scripts/prepare_parking_v1_training.py'
CONFIG = MOBILE/'configs/parking_v1_training_inputs_v1.json'
for path in [SCRIPT, CONFIG, MOBILE/'tests/test_parking_v1_training_inputs.py']:
    if not path.is_file(): raise FileNotFoundError(f'{path} is missing; reload the published main notebook.')
config = json.loads(CONFIG.read_text())
if config['revision'] != REVISION: raise RuntimeError('Notebook/config revision mismatch; reload main.')
requirements = [name+'=='+version for name, version in config['runtime_packages'].items()]
run([PYTHON,'-m','pip','install','--quiet','--upgrade','--only-binary=:all:','--no-deps','--target',DEPS,*requirements], 'cpu_dependencies', MOBILE, task_env=False)
for pattern in ['test_parking_v1_training_inputs.py','test_parking_v0_v1_benchmark.py']:
    run([PYTHON,'-m','unittest','discover','-s','tests','-p',pattern,'-v'], 'tests_'+pattern, MOBILE)

def stage(name):
    command = [PYTHON,'-X','faulthandler','-u',SCRIPT,name,'--dataset',DATASET,'--coverage',COVERAGE,'--output',OUTPUT,'--local',LOCAL,'--workers',str(WORKERS)]
    for path in COVERAGE_UPLOADS: command += ['--coverage-archive',path]
    if DOWNLOAD_MANIFEST_UPLOAD.is_file(): command += ['--download-manifest',DOWNLOAD_MANIFEST_UPLOAD]
    if ALLOW_ONE_TIME_SMALL_DRIVE_READS: command += ['--allow-one-time-small-drive-reads']
    return run(command, 'v1_inputs_'+name, MOBILE)

READY = True
print('CPU preparation is ready. No training or source sensor work has started.')


## 3. Cache coverage metadata and freeze the training selection
If uploading the coverage folder ZIP after section 1, rerun section 1 to refresh `COVERAGE_UPLOADS`, then section 2. Same-VM resumes use the local snapshot. After a reset, one saved `source_state.zip` and one `selection_state.zip` restore the plan.

This stage checks the original 34 training-log assignments and selects at most 4,096 training images without predictions. Dev/held-out views are excluded. Missing near-range diversity stops before sensor preparation; camera-specific gaps and sub-metre scarcity remain visible.

In [ ]:
if not READY: raise RuntimeError('Run sections 1 and 2 first')
stage('snapshot')
stage('plan')
selection = json.loads((LOCAL/'selection_manifest.json').read_text())
pool = selection['selected_pool']
print('Selected:',pool['images'],'views;',pool['unique_keyframes'],'keyframes;',pool['recording_logs'],'training logs')
for cell in config['selection_cells']:
    row = pool['cells'][cell]
    print(cell, '| observations:',row['annotation_observations'],'| tracks:',row['instance_tracks'],'| logs:',row['recording_logs'],'| points:',row['eligible_reference_points'])
for channel, row in pool['by_camera'].items():
    print(channel,'| 0-2m tracks:',{cls:row['cells'][cls+':0-2m']['instance_tracks'] for cls in ['Vehicle','Pedestrian']})
if not pool['six_cell_data_ready']: raise RuntimeError('Data-diversity check failed. Send the selection summary; do not stage sensors or activate a GPU.')


## 4. Prepare RGB + near-depth + broad-protection archives
This is the long **CPU** stage. It copies and verifies one source archive at a time, extracts only selected training RGB/LiDAR, and recreates the reviewed near-depth masks from original measurements. Separate conservative 2–80 m sparse references are retained for later broad-loss protection. No model sees the data here.

Completed source parts are saved as `train_parts/train_partNN.zip` on Drive. After a cell stop, rerun with the same `RUN_ID`; valid completed parts are reused. An incomplete source transfer/shard can resume on the same VM. After VM deletion only completed Drive archives survive; an unfinished part may need rebuilding. Do not run two notebooks with the same `RUN_ID` concurrently.

The snapshot method reduces small-I/O overhead but cannot remove Drive bandwidth limits. On I/O error 5, preserve the files and wait for recovery; do not loop remounts or restarts.

In [ ]:
if not READY: raise RuntimeError('Run sections 1 and 2 first')
stage('prepare')
report = json.loads((LOCAL/'preparation_report.json').read_text())
assert report['complete'] and report['training_performed'] is False
assert report['optimizer_steps']==0 and report['model_inference_performed'] is False
assert report['dev_or_heldout_sensor_samples_exported']==0 and report['phone_app_changed'] is False
print('Verified:',report['images'],'training views in',report['parts'],'part archives; GB:',round(report['archive_bytes']/1e9,3))
print('Training data ready:',report['training_data_ready'])
for warning in report['warnings']: print('NOTE:',warning)
print('Preparation only. Keep the GPU off until this report is reviewed.')


## 5. Download the small preparation report
Send `parking_v1_training_inputs_results.zip` back for review. It contains the frozen training manifest, coverage, plan and identities—not RGB or model weights. The reusable train-part archives stay on Drive. This notebook never starts V1 training.

In [ ]:
if not READY: raise RuntimeError('Run sections 1 and 2 first')
stage('bundle')
BUNDLE = LOCAL/'parking_v1_training_inputs_results.zip'
print('Return this report:',BUNDLE,'| bytes:',BUNDLE.stat().st_size)
from google.colab import files
files.download(str(BUNDLE))
